# LucidRT Interp — What did the fine-tune change? (base vs LoRA)

**Goal:** localize the **+37% ROUGE-L** gain from the LoRA fine-tune. The base model is frozen; the entire fine-tune is a 20.2 MB adapter (r=8, α=16, on q/k/v/o_proj, 0.14% of params). So the change is *small and localized* — which means it's actually findable.

We answer three questions:
1. **Where** did the weights change most? (per-layer / per-projection ‖ΔW‖)
2. **How** did internal representations shift? (base-vs-FT activation drift, per layer)
3. **When** does the model decide on plain language? (logit lens — does it commit earlier after fine-tuning?)

Runs on a free Colab **T4**. §1 re-creates the adapter (we don't have it saved); after that, fill in the `# TODO(you)` lines — hints are right above each one.

## 0. Setup

In [ ]:
# Colab: install the libs. (Skip this line if you're on the cluster env with them already.)
!pip install -q transformers peft accelerate datasets textstat
# Colab ships an old torchao (0.10) that peft's import check rejects -> ImportError at
# get_peft_model in Section 1. We don't need torchao for LoRA, so remove it. If you STILL
# see the error after running this cell, do Runtime > Restart session, then run from here.
!pip uninstall -y -q torchao

import torch, numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_DIR = "lucidrt-lora"   # where the fine-tuned adapter will be written / loaded from
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE=="cuda" else "cpu only")

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

## 1. (Re)create the LoRA adapter

We don't have the adapter saved, so this cell fine-tunes it from scratch — 300 steps, ~a few minutes on a T4. It's the condensed version of Section 2 of `lucidrt_data_pipeline.ipynb`: load the medical-simplification data, wrap each row in the prompt template, inject a LoRA adapter into the attention layers, and train. It saves to `ADAPTER_DIR`.

The cell **skips training if an adapter is already there**, so it's safe to re-run. (If you later find your saved copy, just drop it in `lucidrt-lora/` — or mount Drive and point `ADAPTER_DIR` at it — and this cell becomes a no-op.)

In [ ]:
import os
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from torch.amp import autocast, GradScaler

if os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")):
    print("Adapter already exists at", ADAPTER_DIR, "— skipping training.")
else:
    print("No adapter found — fine-tuning 300 steps to recreate it...")

    # --- data: each row is query (dense medical text) -> answer (plain language) ---
    raw = load_dataset("lighteval/med_paragraph_simplification")
    PROMPT_TEMPLATE = (
        "Simplify the following medical text into plain language a patient can understand.\n\n"
        "Medical text:\n{technical}\n\nPlain language:\n"
    )
    MAX_LEN = 512

    # At TRAINING time we fill BOTH blanks (prompt + answer + eos), so the model learns to
    # produce the plain version after the 'Plain language:' cue. (At generation time we'd
    # fill only {technical} and let it generate the rest.)
    class MedSimplifyDataset(Dataset):
        def __init__(self, split): self.data = split
        def __len__(self): return len(self.data)
        def __getitem__(self, idx):
            ex = self.data[idx]
            text = PROMPT_TEMPLATE.format(technical=ex["query"]) + ex["answer"] + tok.eos_token
            enc = tok(text, truncation=True, max_length=MAX_LEN)
            return {"input_ids": torch.tensor(enc["input_ids"]),
                    "attention_mask": torch.tensor(enc["attention_mask"])}

    @dataclass
    class PadCollator:   # pad a batch of different-length examples into one rectangle
        pad_token_id: int
        def __call__(self, batch):
            max_len = max(e["input_ids"].size(0) for e in batch)
            ids, masks = [], []
            for e in batch:
                pad = max_len - e["input_ids"].size(0)
                ids.append(torch.cat([e["input_ids"], torch.full((pad,), self.pad_token_id, dtype=torch.long)]))
                masks.append(torch.cat([e["attention_mask"], torch.zeros(pad, dtype=torch.long)]))
            return {"input_ids": torch.stack(ids), "attention_mask": torch.stack(masks)}

    train_dl = DataLoader(MedSimplifyDataset(raw["train"]), batch_size=4, shuffle=True,
                          collate_fn=PadCollator(tok.pad_token_id))

    # --- model + LoRA adapter (r=8, alpha=16, injected into attention q/k/v/o) ---
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="cuda")
    lora_cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
                          target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], task_type="CAUSAL_LM")
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()   # ~2.18M trainable (0.14%)

    # memory savers — these are what stop the 12 GB OOM during training
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    model.config.use_cache = False

    # --- 300-step training loop ---
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4)
    scaler = GradScaler("cuda")
    model.train()

    def make_labels(input_ids, attn):
        labels = input_ids.clone()
        labels[attn == 0] = -100   # -100 tells the loss to IGNORE padding positions
        return labels

    MAX_STEPS, step = 300, 0
    while step < MAX_STEPS:
        for batch in train_dl:
            batch = {k: v.to("cuda") for k, v in batch.items()}
            labels = make_labels(batch["input_ids"], batch["attention_mask"])
            opt.zero_grad()
            with autocast("cuda", dtype=torch.float16):
                loss = model(input_ids=batch["input_ids"],
                             attention_mask=batch["attention_mask"], labels=labels).loss
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            step += 1
            if step % 25 == 0:
                print(f"step {step:3d} | loss {loss.item():.3f}")
            if step >= MAX_STEPS:
                break

    # --- save just the adapter (tiny — the 1.5B base is untouched) ---
    model.save_pretrained(ADAPTER_DIR)
    tok.save_pretrained(ADAPTER_DIR)
    mb = sum(os.path.getsize(os.path.join(ADAPTER_DIR, f)) for f in os.listdir(ADAPTER_DIR)) / 1e6
    print(f"saved adapter to {ADAPTER_DIR} ({mb:.1f} MB)")

    # free training memory so Section 2 can reload clean copies
    del model, opt, scaler
    torch.cuda.empty_cache()

## 2. Load both models

We keep two models in memory: the **frozen base** and the **fine-tuned** (base + adapter). Everything below compares them. On a T4 two 1.5B fp16 models is ~6 GB — fine.

In [ ]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16).to(DEVICE).eval()

# Fine-tuned = base + adapter. merge_and_unload() folds the delta into the weights so it
# behaves like a normal model (handy for activation hooks later).
ft = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16).to(DEVICE)
ft = PeftModel.from_pretrained(ft, ADAPTER_DIR)
ft = ft.merge_and_unload().eval()

print("base and fine-tuned models loaded.")

## 2.5 Behavioral check — *what* changed, and did it get simpler?

Before digging into weights, read the two models side by side. ROUGE (from the inference project) only scores **word-overlap with a reference** — it can't tell you whether the text actually got *easier to read*. So we do two things here:

1. **Eyeball** base vs fine-tuned output on the same medical passages — this gives you a concrete *behavioral hypothesis* (shorter sentences? less jargon? defines terms?) to go looking for in §3–§6.
2. **Measure** readability with Flesch–Kincaid grade level (lower = simpler) — the direct "did it simplify?" number ROUGE can't give.

Keep the test **in-domain** (clinical text — what it was fine-tuned on). Out-of-domain text like an insurance policy is a separate *"does it generalize?"* experiment; don't mix the two.

> After running this, write one sentence: **what visibly changed?** That's the hypothesis you'll test against the ‖ΔW‖ heatmap (§3) and the activation drift (§5).

In [ ]:
# Behavioral check: generate from BOTH models on the same prompts, then measure readability.
import textstat, statistics

PROMPT_TEMPLATE = (
    "Simplify the following medical text into plain language a patient can understand.

"
    "Medical text:
{technical}

Plain language:
"
)

# In-domain (medical) passages — the honest test for this fine-tune.
EVAL_SAMPLES = [
    "Two trials met the inclusion criteria. Both were small, reported results inadequately, "
    "and were of low methodological quality, giving very-low certainty evidence.",
    "The intervention group showed a statistically significant reduction in systolic blood "
    "pressure relative to placebo at twelve weeks (p < 0.05).",
    "Participants received topical application weekly for six weeks; the primary endpoint was "
    "complete wound epithelialization at follow-up.",
]

@torch.no_grad()
def generate(model, technical, max_new_tokens=120):
    prompt = PROMPT_TEMPLATE.format(technical=technical)
    enc = tok(prompt, return_tensors="pt").to(DEVICE)
    # TODO(you): greedy-generate, then return ONLY the newly generated text (strip the prompt).
    #   Hint 1: out = model.generate(**enc, max_new_tokens=max_new_tokens,
    #                                do_sample=False, pad_token_id=tok.eos_token_id)
    #   Hint 2: the prompt is the first enc["input_ids"].shape[1] tokens of out[0];
    #           slice those off, then tok.decode(rest, skip_special_tokens=True).strip()
    gen_ids = None   # <-- TODO: the generated-only token ids (a 1-D tensor)
    return tok.decode(gen_ids, skip_special_tokens=True).strip()

rows = []
for tech in EVAL_SAMPLES:
    base_out = generate(base, tech)
    ft_out   = generate(ft,   tech)

    # TODO(you): Flesch-Kincaid US grade level for each output (lower = simpler).
    #   Hint: textstat.flesch_kincaid_grade(some_text) -> float
    base_grade = None   # <-- TODO
    ft_grade   = None   # <-- TODO

    rows.append((base_grade, ft_grade))
    print("=" * 88)
    print("MEDICAL:", tech)
    print(f"
BASE (grade {base_grade}):
  {base_out}")
    print(f"
FT   (grade {ft_grade}):
  {ft_out}")

# Summary: did the fine-tune lower the average reading grade?
base_avg = statistics.mean(r[0] for r in rows)
ft_avg   = statistics.mean(r[1] for r in rows)
print("
" + "=" * 88)
print(f"mean Flesch-Kincaid grade  |  base {base_avg:.1f}  ->  fine-tuned {ft_avg:.1f}  "
      f"(delta {ft_avg - base_avg:+.1f})")

# Bonus (optional, run only after the above works): paste an insurance-policy paragraph as an
# OUT-OF-DOMAIN generalization probe. Report it separately; it is NOT part of the fine-tune eval.
# print(generate(ft, "<insurance paragraph here>"))

## 3. Weight-delta ‖BA‖ — *where* did the weights change?

For each adapted module, the update is **ΔW = (α/r)·B·A** (here α/r = 16/8 = **2.0**). A stores as `lora_A.weight` shape `[r, in]`, B as `lora_B.weight` shape `[out, r]`, so `B@A` is `[out, in]` — the same shape as the base weight.

We measure each module's **‖ΔW‖ / ‖W_base‖** (relative Frobenius norm): how big is the nudge compared to the weight it sits on? This is pure linear algebra — no GPU, no forward pass.

> We reload a *fresh* PeftModel here (not merged) so the separate A/B matrices are still accessible.

In [ ]:
from peft import PeftModel
import re

probe = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32)
probe = PeftModel.from_pretrained(probe, ADAPTER_DIR)

SCALING = 16 / 8   # alpha / r
rows = []
for name, mod in probe.named_modules():
    # a LoRA-wrapped linear has .lora_A and .lora_B submodules
    if hasattr(mod, "lora_A") and hasattr(mod, "lora_B"):
        A = mod.lora_A["default"].weight.data            # [r, in]
        B = mod.lora_B["default"].weight.data            # [out, r]
        Wb = mod.base_layer.weight.data                  # [out, in]
        # TODO(you): compute dW = SCALING * (B @ A), then rel = ||dW|| / ||Wb||
        #   Hint: '@' is matrix multiply; X.norm() is the Frobenius norm of X.
        dW = None   # <-- TODO
        rel = None  # <-- TODO

        layer = int(re.search(r"layers\.(\d+)\.", name).group(1))
        proj = re.search(r"(q|k|v|o)_proj", name).group(1)
        rows.append((layer, proj, float(dW.norm()), float(rel)))

rows.sort()
print(f"{'layer':>5} {'proj':>5} {'||dW||':>10} {'||dW||/||W||':>13}")
for layer, proj, nrm, rel in rows:
    print(f"{layer:>5} {proj:>5} {nrm:>10.3f} {rel:>13.4f}")

In [ ]:
# Visualize: relative update strength as a layer × projection heatmap.
import matplotlib.pyplot as plt

projs = ["q", "k", "v", "o"]
n_layers = max(r[0] for r in rows) + 1
grid = np.zeros((len(projs), n_layers))
for layer, proj, nrm, rel in rows:
    grid[projs.index(proj), layer] = rel

fig, ax = plt.subplots(figsize=(11, 2.6))
im = ax.imshow(grid, aspect="auto", cmap="viridis")
ax.set_yticks(range(len(projs))); ax.set_yticklabels([p + "_proj" for p in projs])
ax.set_xlabel("layer"); ax.set_title("Relative LoRA update  ||dW|| / ||W||")
fig.colorbar(im, ax=ax, shrink=0.8); plt.tight_layout(); plt.show()

# TODO(you): one sentence — which layers / projections carry the biggest change? Early, middle, or late?

## 4. Effective rank of ΔW — is the update concentrated?

The adapter is rank-8 by construction, but the *effective* rank (how many singular values actually matter) can be lower. If ΔW is dominated by 1–2 directions, the fine-tune learned a very specific transform.

In [ ]:
def effective_rank(dW):
    # participation ratio of the singular values: (Σσ)^2 / Σ(σ^2). Ranges 1..r.
    s = torch.linalg.svdvals(dW.float())
    # TODO(you): return float((s.sum()**2) / (s**2).sum())
    return None

# Pick the few highest-||dW|| modules from section 3 and report their effective rank.
# TODO(you): loop over the top-5 rows (by ||dW||), recompute dW for each, print effective_rank(dW).
#   Hint: refactor the dW = SCALING * (B @ A) computation into a small helper so §3 and §4 share it.

## 5. Activation drift — *how* did representations shift?

Feed the **same** medical prompts through base and fine-tuned, capture each layer's hidden state, and measure how different they are per layer. A natural metric is **1 − cosine similarity** of the last-token hidden state (simple), or linear **CKA** (more robust). Expect drift to grow through the stack and peak where the fine-tune did its work.

In [ ]:
PROMPT_TEMPLATE = (
    "Simplify the following medical text into plain language a patient can understand.\n\n"
    "Medical text:\n{technical}\n\nPlain language:\n"
)
SAMPLES = [
    "Two trials met the inclusion criteria. Both were small, reported results inadequately, "
    "and were of low methodological quality, giving very-low certainty evidence.",
    "The intervention group showed a statistically significant reduction in systolic blood "
    "pressure relative to placebo at twelve weeks (p < 0.05).",
    "Participants received topical application weekly for six weeks; the primary endpoint was "
    "complete wound epithelialization at follow-up.",
]
prompts = [PROMPT_TEMPLATE.format(technical=t) for t in SAMPLES]

@torch.no_grad()
def layer_hidden_states(model, text):
    enc = tok(text, return_tensors="pt").to(DEVICE)
    out = model(**enc, output_hidden_states=True)
    # out.hidden_states is a tuple (embeddings, layer1, ..., layerN); take the LAST token of each
    return torch.stack([h[0, -1, :].float() for h in out.hidden_states])   # [n_layers+1, d]

# TODO(you): for each prompt, get hidden states from base and ft; per layer compute
#   cosine similarity, then drift = 1 - cosine. Average drift over the prompts.
#   Hint: torch.nn.functional.cosine_similarity(a, b, dim=0)
drift = None   # <-- TODO: array of length n_layers+1

plt.figure(figsize=(9, 3))
plt.plot(drift, marker="o")
plt.xlabel("layer (0 = embeddings)"); plt.ylabel("1 − cosine (base vs FT)")
plt.title("Representation drift from fine-tuning, per layer"); plt.tight_layout(); plt.show()

## 6. Logit lens — *when* does the model decide on plain language?

The logit lens projects each layer's residual stream through the final `lm_head` (un-embedding) to ask: *if the model had to answer from this layer, what token would it pick?* If the fine-tune helps the model **commit to plain-language words earlier** in the stack, that's a concrete, visual story.

Pick a position (e.g. the first generated token) and track the probability of a plain-language target token across layers, for base vs FT.

In [ ]:
@torch.no_grad()
def logit_lens(model, text, target_token_id):
    enc = tok(text, return_tensors="pt").to(DEVICE)
    out = model(**enc, output_hidden_states=True)
    norm = model.model.norm        # Qwen final RMSNorm before lm_head
    head = model.lm_head
    probs_per_layer = []
    for h in out.hidden_states:                      # each [1, seq, d]
        logits = head(norm(h[:, -1, :]))             # project the last position
        p = torch.softmax(logits.float(), dim=-1)[0, target_token_id]
        probs_per_layer.append(float(p))
    return probs_per_layer

# TODO(you): choose a plain-language target word the FT model prefers (e.g. ' studies', ' showed'),
#   get its token id with tok(' studies').input_ids[-1], then plot logit_lens() for base vs ft
#   on the same prompt. Does the FT curve rise earlier / higher?
#   Hint: compare against a jargon synonym to make the contrast sharp.

## 7. Synthesis — where does the +37% live?

Tie the three views together in a few sentences (this becomes the interview story / a short section in `results/`):

- **Where (§3–4):** which layers/projections changed most, and how low-rank the change is.
- **How (§5):** where representations diverge — does drift track the high-‖ΔW‖ layers?
- **When (§6):** does the fine-tuned model commit to plain language earlier in the stack?

> *TODO(you): write the 3–4 sentence finding here.*